In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


# STEP 1  EXTRACT: Load raw data from the database into Python

In [ ]:

# 1.1 Connect to database and join all 3 tables (bookings + customers + reservations) using SQL then load the result into a pandas DataFrame

conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

df = pd.read_sql_query("""
    SELECT 
        b.booking_id,
        b.hotel,
        b.is_canceled,
        b.lead_time,
        b.arrival_date_year,
        b.arrival_date_month,
        b.arrival_date_week_number,
        b.arrival_date_day_of_month,
        b.stays_in_weekend_nights,
        b.stays_in_week_nights,
        c.adults,
        c.children,
        c.babies,
        b.adr,
        b.booking_changes,
        b.days_in_waiting_list,
        b.required_car_parking_spaces,
        b.total_of_special_requests,
        b.reservation_status,
        b.reservation_status_date,
        c.country,
        c.is_repeated_guest,
        c.previous_cancellations,
        c.previous_bookings_not_canceled,
        r.meal,
        r.market_segment,
        r.distribution_channel,
        r.reserved_room_type,
        r.assigned_room_type,
        r.deposit_type,
        r.agent,
        r.company,
        r.customer_type
    FROM bookings b
    JOIN customers c ON b.booking_id = c.booking_id
    JOIN reservations r ON b.booking_id = r.booking_id
""", conn)

conn.close()

# STEP 2 INSPECT: Understand the raw data before touching anything

In [ ]:
# 2.1 Basic information

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nData types:")
print(df.dtypes)

Rows:    119,390
Columns: 33

Data types:
booking_id                          int64
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
adr                               float64
booking_changes                     int64
days_in_waiting_list                int64
required_car_parking_spaces         int64
total_of_special_requests           int64
reservation_status                    str
reservation_status_date               str
country                               str
is_repeated_guest                   int64
previous_cancellations            

In [ ]:
# 2.2 Null values 

null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct
})
null_df = null_df[null_df['null_count'] > 0]
print(null_df.to_string())

          null_count  null_pct
children           4      0.00
country          488      0.41
agent          16340     13.69
company       112593     94.31


In [40]:
# 2.3 Duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
print(f"Duplicate percentage: {duplicates / len(df) * 100:.2f}%")

Duplicate rows: 0
Duplicate percentage: 0.00%


In [41]:
# 2.4 Statistical summary
print(df.describe())

       booking_id  is_canceled  lead_time  arrival_date_year  \
count   119390.00    119390.00  119390.00          119390.00   
mean     59694.50         0.37     104.01            2016.16   
std      34465.07         0.48     106.86               0.71   
min          0.00         0.00       0.00            2015.00   
25%      29847.25         0.00      18.00            2016.00   
50%      59694.50         0.00      69.00            2016.00   
75%      89541.75         1.00     160.00            2017.00   
max     119389.00         1.00     737.00            2017.00   

       arrival_date_week_number  arrival_date_day_of_month  \
count                 119390.00                  119390.00   
mean                      27.17                      15.80   
std                       13.61                       8.78   
min                        1.00                       1.00   
25%                       16.00                       8.00   
50%                       28.00                    

In [ ]:
# 2.5.1 Anomaly investigation

zero_adults = df[df['adults'] == 0]
print(f"Bookings with 0 adults: {len(zero_adults)}")
print(zero_adults[['hotel', 'adults', 'children', 'babies', 
                    'adr', 'reservation_status']].head(10))

print()

extreme_adults = df[df['adults'] > 10]
print(f"\nBookings with more than 10 adults: {len(extreme_adults)}")
print(extreme_adults[['hotel', 'adults', 'children', 
                       'babies', 'adr', 'market_segment']].head(10))

print()


Bookings with 0 adults: 403
              hotel  adults  children  babies   adr reservation_status
2224   Resort Hotel       0      0.00       0  0.00          Check-Out
2409   Resort Hotel       0      0.00       0  0.00          Check-Out
3181   Resort Hotel       0      0.00       0  0.00          Check-Out
3684   Resort Hotel       0      0.00       0  0.00          Check-Out
3708   Resort Hotel       0      0.00       0  0.00          Check-Out
4127   Resort Hotel       0      0.00       0  0.00           Canceled
9376   Resort Hotel       0      0.00       0  0.00           Canceled
31765  Resort Hotel       0      0.00       0 28.00          Check-Out
32029  Resort Hotel       0      0.00       0  0.00          Check-Out
32827  Resort Hotel       0      0.00       0  0.00          Check-Out


Bookings with more than 10 adults: 12
             hotel  adults  children  babies  adr market_segment
1539  Resort Hotel      40      0.00       0 0.00         Direct
1587  Resort Hotel   

In [ ]:
# 2.5.2 Anomaly investigation

print(f"\nADR statistics:")
print(f"Min ADR:  ${df['adr'].min():.2f}")
print(f"Max ADR:  ${df['adr'].max():.2f}")
print(f"Mean ADR: ${df['adr'].mean():.2f}")
print(f"\nBookings with ADR = 0: {len(df[df['adr'] == 0])}")
print(f"Bookings with ADR > 500: {len(df[df['adr'] > 500])}")

print()

zero_nights = df[(df['stays_in_weekend_nights'] == 0) & 
                 (df['stays_in_week_nights'] == 0)]
print(f"Bookings with 0 total nights: {len(zero_nights)}")
print(zero_nights[['hotel', 'adr', 'reservation_status']].head(5))


ADR statistics:
Min ADR:  $-6.38
Max ADR:  $5400.00
Mean ADR: $101.83

Bookings with ADR = 0: 1959
Bookings with ADR > 500: 3

Bookings with 0 total nights: 715
            hotel  adr reservation_status
0    Resort Hotel 0.00          Check-Out
1    Resort Hotel 0.00          Check-Out
167  Resort Hotel 0.00          Check-Out
168  Resort Hotel 0.00          Check-Out
196  Resort Hotel 0.00          Check-Out


In [ ]:
# 2.5.3 Anomaly investigation

print(f"ADR = 0:      {len(df[df['adr'] == 0]):,} bookings")
print(f"ADR > 500:    {len(df[df['adr'] > 500]):,} bookings")
print(f"ADR > 1000:   {len(df[df['adr'] > 1000]):,} bookings")
print(f"Max ADR:      ${df['adr'].max():.2f}")

# Show extreme ADR rows
print("\nTop 5 highest ADR bookings:")
print(df.nlargest(5, 'adr')[['hotel', 'adr', 'adults', 
                              'stays_in_week_nights',
                              'stays_in_weekend_nights',
                              'market_segment', 
                              'reservation_status']])

print()



# Zero nights full picture
zero_nights = df[(df['stays_in_weekend_nights'] == 0) & 
                 (df['stays_in_week_nights'] == 0)]
print(f"Bookings with 0 total nights: {len(zero_nights):,}")
print(f"Of these, how many checked out: {len(zero_nights[zero_nights['reservation_status'] == 'Check-Out']):,}")
print(f"Of these, how many canceled: {len(zero_nights[zero_nights['reservation_status'] == 'Canceled']):,}")

ADR = 0:      1,959 bookings
ADR > 500:    3 bookings
ADR > 1000:   1 bookings
Max ADR:      $5400.00

Top 5 highest ADR bookings:
               hotel     adr  adults  stays_in_week_nights  \
48515     City Hotel 5400.00       2                     1   
111403    City Hotel  510.00       1                     1   
15083   Resort Hotel  508.00       2                     1   
103912    City Hotel  451.50       2                     1   
13142   Resort Hotel  450.00       2                    10   

        stays_in_weekend_nights market_segment reservation_status  
48515                         0  Offline TA/TO           Canceled  
111403                        0  Offline TA/TO          Check-Out  
15083                         0      Corporate          Check-Out  
103912                        1         Direct          Check-Out  
13142                         4      Online TA           Canceled  

Bookings with 0 total nights: 715
Of these, how many checked out: 680
Of these, how man

# STEP 3 TRANSFORM: Fix all data quality issues identified in Step 2

In [ ]:
# 3.1 Fix null values

df_clean = df.copy()

df_clean['children'] = df_clean['children'].fillna(0)
print(f"  children nulls : {df_clean['children'].isnull().sum()}")

df_clean['country'] = df_clean['country'].fillna('Unknown')
print(f"  country nulls : {df_clean['country'].isnull().sum()}")

df_clean['agent'] = df_clean['agent'].fillna(0)
print(f"  agent nulls : {df_clean['agent'].isnull().sum()}")

df_clean['company'] = df_clean['company'].fillna(0)
print(f"  company nulls : {df_clean['company'].isnull().sum()}")


  children nulls : 0
  country nulls : 0
  agent nulls : 0
  company nulls : 0


In [54]:
# 3.2 Fix data types

df_clean['children'] = df_clean['children'].astype(int)
df_clean['agent'] = df_clean['agent'].astype(int)
df_clean['company'] = df_clean['company'].astype(int)


df_clean['reservation_status_date'] = pd.to_datetime(
    df_clean['reservation_status_date']
)

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
df_clean['arrival_date_month'] = pd.Categorical(
    df_clean['arrival_date_month'],
    categories=month_order,
    ordered=True
)

print(f"\n  children dtype:  {df_clean['children'].dtype}")
print(f"  agent dtype:     {df_clean['agent'].dtype}")
print(f"  company dtype:   {df_clean['company'].dtype}")
print(f"  date dtype:      {df_clean['reservation_status_date'].dtype}")
print(f"  month dtype:     {df_clean['arrival_date_month'].dtype}")


  children dtype:  int64
  agent dtype:     int64
  company dtype:   int64
  date dtype:      datetime64[us]
  month dtype:     category


In [63]:
# 3.3 Handle outliers and flag anomalies:

# 3.3.1 Flag zero-guest bookings
df_clean['is_zero_guest'] = (df_clean['adults'] == 0).astype(int)
print(f"  Flagged zero-guest bookings: {df_clean['is_zero_guest'].sum()}")

# 3.3.2 Cap adults at 10 
df_clean['adults'] = df_clean['adults'].clip(upper=10)
print(f"  Max adults: {df_clean['adults'].max()}")

# 3.3.3 Fix negative ADR 
neg_adr_count = len(df_clean[df_clean['adr'] < 0])
print(f"\n  Rows with negative ADR : {neg_adr_count}")
print(f"  Negative ADR value: {df_clean[df_clean['adr'] < 0]['adr'].values}")
df_clean.loc[df_clean['adr'] < 0, 'adr'] = 0
print(f"  Min ADR : ${df_clean['adr'].min():.2f}")

# 3.3.4 Flag complimentary stays 
df_clean['is_complimentary'] = (df_clean['adr'] == 0).astype(int)
print(f"\n  Flagged complimentary stays: {df_clean['is_complimentary'].sum()}")

# 3.3.5 Cap ADR at 500 
df_clean['adr'] = df_clean['adr'].clip(upper=500)
print(f"  Max ADR: ${df_clean['adr'].max():.2f}")

# 3.3.6 Flag same-day bookings
df_clean['is_same_day'] = (
    (df_clean['stays_in_weekend_nights'] == 0) & 
    (df_clean['stays_in_week_nights'] == 0)
).astype(int)
print(f"\n  Flagged same-day bookings: {df_clean['is_same_day'].sum()}")



  Flagged zero-guest bookings: 403
  Max adults: 10

  Rows with negative ADR : 0
  Negative ADR value: []
  Min ADR : $0.00

  Flagged complimentary stays: 1960
  Max ADR: $500.00

  Flagged same-day bookings: 715


# STEP 4 FEATURE ENGINEERING: Create new columns that power analysis and ML

In [66]:
# 4.1 total_nights — weekend + weekday night

df_clean['total_nights'] = (
    df_clean['stays_in_weekend_nights'] + 
    df_clean['stays_in_week_nights']
)
print(f"total_nights created. Range: {df_clean['total_nights'].min()}–{df_clean['total_nights'].max()}")



# 4.2 total_revenue — ADR × total nights

df_clean['total_revenue'] = (
    df_clean['adr'] * df_clean['total_nights']
).round(2)
print(f"total_revenue created. Range: ${df_clean['total_revenue'].min():.2f}–${df_clean['total_revenue'].max():.2f}")


# 4.3 arrival_date — proper datetime from year/month/day

df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_date_month'].astype(str) + '-' +
    df_clean['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)
print(f"arrival_date created. Range: {df_clean['arrival_date'].min().date()} to {df_clean['arrival_date'].max().date()}")


#4.4 is_weekend_arrival — arrival on Saturday or Sunday

df_clean['is_weekend_arrival'] = (
    df_clean['arrival_date'].dt.dayofweek >= 5
).astype(int)
print(f"is_weekend_arrival created. Weekend arrivals: {df_clean['is_weekend_arrival'].sum():,}")


# 4.5 season — Spring / Summer / Autumn / Winter

def get_season(month):
    if month in ['December', 'January', 'February']:
        return 'Winter'
    elif month in ['March', 'April', 'May']:
        return 'Spring'
    elif month in ['June', 'July', 'August']:
        return 'Summer'
    else:
        return 'Autumn'

df_clean['season'] = df_clean['arrival_date_month'].astype(str).apply(get_season)
print(f"season created. Distribution:\n{df_clean['season'].value_counts().to_string()}")

total_nights created. Range: 0–69
total_revenue created. Range: $0.00–$7590.00
arrival_date created. Range: 2015-07-01 to 2017-08-31
is_weekend_arrival created. Weekend arrivals: 32,196
season created. Distribution:
season
Summer    37477
Spring    32674
Autumn    28462
Winter    20777


In [68]:
# 4.6 lead_time_bucket — 6 booking advance categories

def get_lead_time_bucket(lead_time):
    if lead_time == 0:
        return '1. Same day'
    elif lead_time <= 7:
        return '2. 1-7 days'
    elif lead_time <= 30:
        return '3. 8-30 days'
    elif lead_time <= 90:
        return '4. 31-90 days'
    elif lead_time <= 180:
        return '5. 91-180 days'
    else:
        return '6. 180+ days'

df_clean['lead_time_bucket'] = df_clean['lead_time'].apply(get_lead_time_bucket)
print("lead_time_bucket created:")
print(df_clean['lead_time_bucket'].value_counts().sort_index().to_string())

# 4.7 room_match — reserved room = assigned room?

df_clean['room_match'] = (
    df_clean['reserved_room_type'] == df_clean['assigned_room_type']
).astype(int)
print(f"\nroom_match created.")
print(f"  Room matched: {df_clean['room_match'].sum():,}")
print(f"  Room changed: {(df_clean['room_match'] == 0).sum():,}")

# 4.8 is_family — booking includes children or babies

df_clean['is_family'] = (
    (df_clean['children'] > 0) | 
    (df_clean['babies'] > 0)
).astype(int)
print(f"\nis_family created.")
print(f"  Family bookings:     {df_clean['is_family'].sum():,}")
print(f"  Non-family bookings: {(df_clean['is_family'] == 0).sum():,}")

# 4.9 total_guests — adults + children + babies

df_clean['total_guests'] = (
    df_clean['adults'] + 
    df_clean['children'] + 
    df_clean['babies']
)
print(f"\ntotal_guests created.")
print(f"  Min: {df_clean['total_guests'].min()}")
print(f"  Max: {df_clean['total_guests'].max()}")
print(f"  Mean: {df_clean['total_guests'].mean():.2f}")

# 4.10 revenue_per_night — total revenue ÷ total nights

df_clean['revenue_per_night'] = (
    df_clean['total_revenue'] / 
    df_clean['total_nights'].replace(0, 1)  # avoid division by zero
).round(2)
print(f"\nrevenue_per_night created.")
print(f"  Min: ${df_clean['revenue_per_night'].min():.2f}")
print(f"  Max: ${df_clean['revenue_per_night'].max():.2f}")
print(f"  Mean: ${df_clean['revenue_per_night'].mean():.2f}")


lead_time_bucket created:
lead_time_bucket
1. Same day        6345
2. 1-7 days       13401
3. 8-30 days      18960
4. 31-90 days     29553
5. 91-180 days    26439
6. 180+ days      24692

room_match created.
  Room matched: 104,473
  Room changed: 14,917

is_family created.
  Family bookings:     9,332
  Non-family bookings: 110,058

total_guests created.
  Min: 0
  Max: 12
  Mean: 1.97

revenue_per_night created.
  Min: $0.00
  Max: $500.00
  Mean: $101.79


# STEP 5  VALIDATE: Confirm clean data is correct before saving

In [72]:
print(f"\nShape: {df_clean.shape}")
print(f"Total nulls: {df_clean.isnull().sum().sum()}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")
print(f"Negative ADR rows: {len(df_clean[df_clean['adr'] < 0])}")
print(f"Negative revenue rows: {len(df_clean[df_clean['total_revenue'] < 0])}")
print(f"ADR range: ${df_clean['adr'].min():.2f} – ${df_clean['adr'].max():.2f}")
print(f"Arrival date range: {df_clean['arrival_date'].min().date()} to {df_clean['arrival_date'].max().date()}")
print(f"Cancellation rate: {df_clean['is_canceled'].mean()*100:.2f}%")
print(f"  Seasons:        {df_clean['season'].value_counts().to_dict()}")
print(f"  Room match:     {df_clean['room_match'].value_counts().to_dict()}")
print(f"  Family:         {df_clean['is_family'].value_counts().to_dict()}")
print(f"  Weekend arrival:{df_clean['is_weekend_arrival'].value_counts().to_dict()}")


Shape: (119390, 46)
Total nulls: 0
Duplicate rows: 0
Negative ADR rows: 0
Negative revenue rows: 0
ADR range: $0.00 – $500.00
Arrival date range: 2015-07-01 to 2017-08-31
Cancellation rate: 37.04%
  Seasons:        {'Summer': 37477, 'Spring': 32674, 'Autumn': 28462, 'Winter': 20777}
  Room match:     {1: 104473, 0: 14917}
  Family:         {0: 110058, 1: 9332}
  Weekend arrival:{0: 87194, 1: 32196}


# STEP 6 LOAD: Save clean data for use in all subsequent phases

In [ ]:
# 6.1 Save to CSV

base_path = r'C:\Users\sadeq\Desktop\Python\Hotel_Booking'
csv_path = f'{base_path}\\data\\hotel_bookings_clean.csv'
df_clean.to_csv(csv_path, index=False)

# 6.2 Save to SQLite database

conn = sqlite3.connect(f'{base_path}\\data\\hotel_bookings.db')
df_clean.to_sql('hotel_bookings_clean', conn, 
                if_exists='replace', index=False)
conn.close()
